**FASS G022:** https://drive.google.com/file/d/1J_hyT1cbTQyFe3AS4ip0-N281i1d4Pi3/view?usp=sharing \
**FENS L045:** https://drive.google.com/file/d/1jxK1VwYxKi4WzGvpM8A2Yw_a8pW6eHrK/view?usp=sharing

# CS415 Lab 3: Machine Translation with Transformer Networks

In this lab, we're going to code a transformer network from scratch, and train it to do machine translation.
Assume we have a source sentence $x = (x_1, x_2, \ldots, x_S)$, $x_s \in V_{\text{src}}$ and a target sentence $y = (y_1, y_2, \ldots, y_T)$, $y_t \in V_{\text{tgt}}$. We want to learn the conditional probability distribution $$p(y \mid x) = p(y_1, \ldots, y_T \mid x) = \Pi_{t=1}^T p(y_t \mid y_{1:t-1}, x).$$

We will be building a decoder-only model, in which we consider the task of translation as causal next token prediction, that is, the context so far will be used to predict the next token. Encoder-decoder models are much more common for this task, but the focus here is on self-attention and auto-regression, which is the principle behind modern LLMs like GPT.

In [ ]:
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from datasets import load_dataset
from transformers import AutoTokenizer

from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Our dataset consists of sentences in German and their English translations. We need three special tokens to indicate the beginning of a sentence, the end of a sentence, and the padding.

In [ ]:
tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en")

if tok.bos_token is None:
    tok.add_special_tokens({"bos_token": "<bos>"})
if tok.pad_token is None:
    tok.add_special_tokens({"pad_token": "<pad>"})
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": "<eos>"})

bos_id = tok.bos_token_id # 0
pad_id = tok.pad_token_id # 1
eos_id = tok.eos_token_id # 2

vocab_size = len(tok)
print("Vocab size:", vocab_size)

ds_train = load_dataset("bentrevett/multi30k", split="train")       # fields: "de", "en"
ds_valid = load_dataset("bentrevett/multi30k", split="validation")

print(len(ds_train), "training examples")
print("Example:", ds_train[1])

class TranslationDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex = self.data[idx]
        return ex["de"], ex["en"]

train_ds = TranslationDataset(ds_train)
valid_ds = TranslationDataset(ds_valid)

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Vocab size: 58102
29000 training examples
Example: {'en': 'Several men in hard hats are operating a giant pulley system.', 'de': 'Mehrere Männer mit Schutzhelmen bedienen ein Antriebsradsystem.'}


We model sequences like [DE tokens] + [BOS] + [EN tokens] + [EOS] for the decoder-only model, and create input/target pairs for next token prediction. We use padding to make all sequences same length, and mask out padding tokens. Also, the model will be trained only to translate into the target language, and not model the source language. Therefore we mask German tokens in the loss.

Your task is to:
* Build the token sequences to be fed into the model
* Create the padding mask so that the model ignores padding tokens
* Create a loss mask that excludes source language tokens from the loss

In [ ]:
def make_collate_fn(tokenizer, block_size=256, train_on_source=False):
    pad_id = tokenizer.pad_token_id
    bos_id = tokenizer.bos_token_id
    eos_id = tokenizer.eos_token_id

    def encode(text):
        return tokenizer(text, add_special_tokens=False)["input_ids"]

    def collate(batch):
        input_list = []
        target_list = []
        end_of_de_list = []

        for de_txt, en_txt in batch:
            de_ids = encode(de_txt)
            en_ids = encode(en_txt)

            seq = de_ids + [bos_id] + en_ids + [eos_id]

            bos_pos = len(de_ids)

            # ensure we can make (input, target) pairs of length <= block_size
            seq = seq[:block_size + 1]

            inp = torch.tensor(seq[:-1], dtype=torch.long)
            tgt = torch.tensor(seq[1:], dtype=torch.long)

            if not train_on_source:
                cut = min(bos_pos, tgt.size(0))
                tgt[:cut] = -100

            input_list.append(inp)
            target_list.append(tgt)
            end_of_de_list.append(min(bos_pos + 1, inp.size(0)))

        input_ids = pad_sequence(input_list, batch_first=True, padding_value=pad_id)
        labels    = pad_sequence(target_list, batch_first=True, padding_value=-100)

        attention_mask = (input_ids != pad_id).long()
        end_of_de = torch.tensor(end_of_de_list, dtype=torch.long)

        return {
            "input_ids": input_ids,        # (B, T)
            "attention_mask": attention_mask,  # (B, T)
            "labels": labels,              # (B, T)
            "end_of_DE": end_of_de,
        }

    return collate

block_size = 128  # context length
train_on_source = False
collate_fn = make_collate_fn(tok, block_size=block_size, train_on_source=train_on_source)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
valid_dl = DataLoader(valid_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

batch = next(iter(train_dl))
print("input_ids shape:", batch["input_ids"].shape)
print("attention_mask shape:", batch["attention_mask"].shape)
print("labels shape:", batch["labels"].shape)

input_ids shape: torch.Size([64, 67])
attention_mask shape: torch.Size([64, 67])
labels shape: torch.Size([64, 67])


At the input layer, we map each token id to an embedding vector, and add positional embeddings to encode the position of each token within the sequence. These embeddings are trainable parameters. We either concatenate the token and positional embeddings, or add them, as we do in this case.

In [ ]:
class TokenPositionalEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_seq_len):
        super().__init__()
        self.token_embd = nn.Embedding(vocab_size, d_model)
        self.pos_embd = nn.Embedding(max_seq_len, d_model)

    def forward(self, input_ids):
        B, T = input_ids.shape # (B, T)
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0)  # (1, T)
        tok = self.token_embd(input_ids)           # (B, T, d_model)
        pos = self.pos_embd(positions)            # (1, T, d_model)
        return tok + pos                         # (B, T, d_model)

## Architecture

### Attention

The attention mechanism enables the model to focus on the most relevant parts of the input sequence.
<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20231212180658/selfattne.png" width="35%"/>
  <br>
  <em>Self Attention</em>
</p>
For an input $X = x_{1:n}$, where each token $x \in \mathbb{R}^d$
, we first obtain queries, keys and values:

\begin{align}
Q &= XW_Q \in \mathbb{R}^{n\times d_k}\\
K &= XW_K \in \mathbb{R}^{n\times d_k}\\
V &= XW_V \in \mathbb{R}^{n\times d_v}\\
\end{align}

For each query ${q_i} \in \mathbb{R}^{d_k}$ we compute its similarity to a key with

\begin{align*}
e_{ij} = \frac{q_i^\top k_j}{\sqrt{d_k}}
\end{align*}
Then we normalize this across all values:
\begin{align*}
\alpha_{ij} = \frac{ \exp(e_{ij}) } {\sum_m \exp(e_{im}) }
\end{align*}
The final output is a weighted average of values:
\begin{align*}
z_i = \sum_j \alpha_{ij} v_j
\end{align*}
<p align="center">
  <img src="https://sebastianraschka.com/images/blog/2023/self-attention-from-scratch/summary.png" width="60%"/>
  <br>
  <em>Self-attention Calculations</em>
</p>
Multi-head self attention splits the model into distinct heads, the outputs of which are then concatenated.
<p align="center">
  <img src="https://media.geeksforgeeks.org/wp-content/uploads/20231212181418/multihead.png" width="35%"/>
  <br>
  <em>Multi-head self attention</em>
</p>
We apply causal masking to prevent the model from looking into the future.
<p align="center">
  <img src="https://cdn.prod.website-files.com/6305e5d52c28356b4fe71bac/66c62286f56241fb5e925707_63f8cfaeb05eed305bbc24f4_Holistic-AI-Figure-1.png" width="45%"/>
  <br>
  <em>Causal vs masked language models</em>
</p>

Your task is to:

* Write the output dimension of each attention head
* Write the code to calculate the attention scores
* Write the code to calculate the final output

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads, causal=True, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model # Embedding Dimension
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.causal = causal

        self.qkv = nn.Linear(d_model, 3 * d_model) # Combined q,k,v projection
        self.out = nn.Linear(d_model, d_model) # Output Projection
        self.attn_dropout = nn.Dropout(dropout)

        self.causal_mask = None
        self.last_attn = None

    def _get_causal_mask(self, T, device): # d*V
        if (
            self.causal_mask is None
            or self.causal_mask.size(0) < T
            or self.causal_mask.device != device
        ):
            # Mask where j > i
            self.causal_mask = torch.triu(
                torch.ones(T, T, device=device, dtype=torch.bool),
                diagonal=1
            )
        return self.causal_mask[:T, :T]

    def forward(self, x, attention_mask=None, store_attn=False):
        B, T, D = x.shape
        H = self.num_heads
        d_h = self.d_head

        qkv = self.qkv(x)             # (B, T, 3D)
        q, k, v = qkv.chunk(3, dim=-1)

        # Split into heads
        q = q.view(B, T, H, d_h).transpose(1, 2)  # (B, H, T, d_h)
        k = k.view(B, T, H, d_h).transpose(1, 2)
        v = v.view(B, T, H, d_h).transpose(1, 2)

        # T, d_h x d_h , T -> T, T
        scores = (q@k.transpose(-2, -1)) / math.sqrt(d_h) #scores[i, j]: How much does token at i attend to token at j

        mask = None
        if self.causal:
            causal_mask = self._get_causal_mask(T, x.device)         # (T, T)
            mask = causal_mask.unsqueeze(0).unsqueeze(0)             # (1, 1, T, T)

        if attention_mask is not None:
            key_mask = (attention_mask == 0).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, T)
            mask = key_mask if mask is None else (mask | key_mask)

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attn = attn.softmax(dim=-1)
        attn = self.attn_dropout(attn)

        if store_attn:
            self.last_attn = attn.detach().cpu()
        else:
            self.last_attn = None

        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, D)  # (B, T, D)
        out = self.out(out)                                   # (B, T, D)
        return out

### Transformer


Each Transformer block has two main components:
- **Multi-Head Self-Attention (MHSA)**, which allows every token to attend to other tokens in the sequence, gathering relevant context.  
- **Feed-Forward Network (FFN)**, a small MLP applied to each token representation independently. Without this we would be stacking linear maps plus softmax.

Both parts are wrapped with **residual connections** and **layer normalization** for stable training:

- **Layer normalization**  works by normalizing the activations of each token representation across its hidden dimensions, ensuring that values have consistent scale and distribution. This helps gradients flow more reliably, reduces sensitivity to initialization, and allows the model to train deeper networks without diverging. In practice, it is usually applied together with **residual connections**, so that the model learns new information while preserving the original signal.


<p align="center">
  <img src="https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/Transformer-neural-network-12.png" width="25%"/>
  <br>
  <em>Transformer network with positional embeddings and transformer blocks (image from https://arxiv.org/abs/1706.03762)</em>
</p>



Your task is to:

* Fill in the correct dimensions in the FFN
* Code the missing steps in the forward pass

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, causal=True):
        super().__init__()
        self.attn = MultiHeadSelfAttention(d_model, num_heads, causal=causal, dropout=dropout)
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attention_mask=None, store_attn=False):
        h = self.ln1(x) # PRE-LAYER NORMALIZATION
        h = self.attn(h, attention_mask=attention_mask, store_attn=store_attn)
        x = x + self.dropout(h) # RESIDUAL CONNECTION

        h2 = self.ln2(x)
        h2 = self.ff(h2)
        x = x + self.dropout(h2)

        return x

These embeddings are then passed through a stack of transformer blocks defined above. Weight tying between the token and head embeddings is used to reduce the number of parameters. This makes senses because they both contain the same information. A token's contextual representation is used to decide when it should appear as the output.

Your task is to:
* Define the language modeling head
* Tie the weights of the language modeling head and the token embeddings matrix
* Code the missing steps in the forward pass

In [ ]:
class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.embed = TokenPositionalEmbedding(vocab_size, d_model, max_seq_len)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout=dropout, causal=True)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)

        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # The language modeling head and the token embeddings share weights
        self.head.weight = self.embed._token_embd.weight

    def forward(self, input_ids, attention_mask=None, store_attn=False):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            input_ids = input_ids[:, -self.max_seq_len:]
            if attention_mask is not None:
                attention_mask = attention_mask[:, -self.max_seq_len:]
            T  = self.max_seq_len

        x = self.embed(input_ids)

        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, store_attn=store_attn)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

## Training

In [ ]:
d_model     = 256
num_heads   = 8
num_layers  = 4
d_ff        = 4 * d_model
max_seq_len = block_size
dropout     = 0.1

model = TransformerLanguageModel(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    num_layers=num_layers,
    d_ff=d_ff,
    max_seq_len=max_seq_len,
    dropout=dropout,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
criterion = nn.CrossEntropyLoss(ignore_index=-100)

checkpoint_path = "decoder_deen"

if os.path.exists(checkpoint_path):
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state)
    print(f"Loaded weights from {checkpoint_path}")
else:
    print("Training from scratch")

print(sum(p.numel() for p in model.parameters()), "parameters")

AttributeError: 'TokenPositionalEmbedding' object has no attribute '_token_embd'

The loss function is masked cross-entropy, i.e. the negative log-likelihood of the correct target tokens. During training we use teacher forcing, i.e. the model conditions on the ground-truth previous tokens instead of its predictions.

$$\mathcal{L}(\theta)
= \frac{1}{N_{\text{valid}}}
\sum_{t \in \mathcal{V}} - \log p_\theta\big(y_t \mid \text{context}_t\big)$$

Your task is to:
* Define the loss function

In [ ]:
def run_epoch(model, dataloader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_tokens = 0

    loop = tqdm(dataloader, desc="train" if train else "eval")
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        if train:
            optimizer.zero_grad()

        logits = model(input_ids, attention_mask=attention_mask)  # (B, T, V)
        B, T, V = logits.shape

        loss = criterion(
            logits.view(B * T, V),
            labels.view(B *T)
        )

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        num_tokens = (labels != -100).sum().item()
        total_loss += loss.item() * num_tokens
        total_tokens += num_tokens

        if total_tokens > 0:
            loop.set_postfix(loss=total_loss / total_tokens)

    return total_loss / max(1, total_tokens)

num_epochs = 20

for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")
    train_loss = run_epoch(model, train_dl, train=True)
    val_loss   = run_epoch(model, valid_dl, train=False)
    print(f"  train NLL/token = {train_loss:.4f}, valid = {val_loss:.4f}")

torch.save(model.state_dict(), checkpoint_path)
print(f"Saved weights to {checkpoint_path}")

## Inference

To generate output from a language model, we can use various inference time algorithms like greedy search or top-k sampling. In practice beam search is used. The temperature parameter is used to smooth out the output distribution, which results in more diverse outputs.

In [ ]:
@torch.no_grad()
def generate(model, input_ids, max_new_tokens, temperature=1.0, top_k=None):
    model.eval()
    for _ in range(max_new_tokens):
        B, T = input_ids.shape
        if T > model.max_seq_len:
            input_ids = input_ids[:, -model.max_seq_len:]

        attention_mask = (input_ids != pad_id).long()
        logits = model(input_ids, attention_mask=attention_mask)  # (B, T, V)
        logits = logits[:, -1, :] / temperature                   # (B, V)

        if top_k is not None:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)      # (B, 1)
        input_ids = torch.cat([input_ids, next_token], dim=1)

        if (next_token == eos_id).all():
            break

    return input_ids

def translate_german_sentence(model, tokenizer, de_text, max_new_tokens=50):
    de_ids = tokenizer(de_text, add_special_tokens=False)["input_ids"]
    seq = de_ids + [bos_id]
    input_ids = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)

    out_ids = generate(model, input_ids, max_new_tokens=max_new_tokens, temperature=1.0, top_k=20)
    out_ids = out_ids[0].tolist()

    if eos_id in out_ids:
        eos_pos = out_ids.index(eos_id)
        out_ids = out_ids[:eos_pos+1]

    de_len = len(de_ids)
    de_part = out_ids[:de_len]
    en_part = out_ids[de_len:]

    return tokenizer.decode(de_part, skip_special_tokens=True), tokenizer.decode(en_part, skip_special_tokens=True)

example_de = ds_valid[0]["de"]
de_out, en_out = translate_german_sentence(model, tok, example_de)
print("Original DE:", example_de)
print("Model DE  :", de_out)
print("Model EN  :", en_out)

Attention visualizations reveal which source words each target word attends to.

In [ ]:
import matplotlib.pyplot as plt

def plot_attention(tokens, attn_matrix):
    T = len(tokens)
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(attn_matrix[:T, :T], aspect="auto", cmap="viridis")
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(T))
    ax.set_yticks(range(T))
    ax.set_xticklabels(tokens, rotation=90)
    ax.set_yticklabels(tokens)
    plt.tight_layout()
    plt.show()

# Visualize attention of the last layer, head 0, on a validation sentence
batch = next(iter(valid_dl))
input_ids = batch["input_ids"][0:1].to(device)
attention_mask = batch["attention_mask"][0:1].to(device)

_ = model(input_ids, attention_mask=attention_mask, store_attn=True)

attn = model.layers[-1].attn.last_attn  # (1, H, T, T)
if attn is not None:
    valid_len = attention_mask[0].sum().item()
    attn_head0 = attn[0, 0, :valid_len, :valid_len]
    tokens = tok.convert_ids_to_tokens(input_ids[0, :valid_len].tolist())
    plot_attention(tokens, attn_head0.numpy())

## Exercises
1. Try removing positional embeddings, or change causal=False in one layer - what happens?
2. Reduce num_heads to 1, or train on the source language as well - how does translation quality change?
3. Visualize attention for different layers - what patterns emerge?
4. Build an encoder-decoder model for this task:
* Encoder: Source tokens as input, several self-attention layers, no causal mask.

* Decoder: Previous target tokens, self-attention with causal mask, cross-attention: queries from decoder, keys/values from encoder outputs.